# Ontology-Aware Knowledge Extraction: From Text to Knowledge Graph

## Overview

This notebook demonstrates **ontology-driven entity extraction** using **GPT-5.2** to transform unstructured text into structured RDF knowledge graphs. Unlike traditional Named Entity Recognition (NER) systems, this approach leverages the semantic understanding of modern LLMs combined with formal ontologies to achieve **concept-aware extraction**.

## The Extreme Challenge: Oliver Twist + Scattered Jaguar Data

The corpus (`oliver_twist_corpus.txt`) presents an **extreme test case** for ontology-aware extraction:

- 📚 **956,000+ characters** of Charles Dickens' complete novel "Oliver Twist"
- 🐆 **Wildlife jaguar conservation data** scattered throughout the text (individual jaguars, locations, organizations, threats)
- 🚗 **Jaguar car mentions** embedded in the corpus
- 🎸 **Fender Jaguar guitar references** mixed in

**The challenge**: Can GPT-5.2 maintain attention across ~250,000 tokens, distinguish wildlife jaguars from cars and guitars, and extract ONLY the conservation-relevant entities aligned to our ontology?

**Spoiler**: Yes. This demonstrates GPT-5.2's remarkable context window and semantic reasoning capabilities.

## The Solution: Ontology-Driven Knowledge Representation

By providing the LLM with our **jaguar conservation ontology** (`jaguar_ontology.ttl`), the model understands:
- The **domain context** (wildlife conservation)
- The **class hierarchy** (Jaguar → Animal → Species)
- The **properties and relationships** (hasGender, occursIn, monitoredByOrg)
- The **semantic structure** of our knowledge domain

The LLM uses this ontological understanding to:
1. **Filter relevant entities** - Only extract wildlife-related jaguars, ignoring cars and guitars
2. **Recognize relationships** - Identify connections between jaguars, locations, organizations, threats
3. **Generate aligned RDF** - Produce Turtle syntax that perfectly matches our ontology structure
4. **Infer implicit relationships** - Connect entities based on contextual understanding

## Why RDF Ontologies (Not Natural Language Descriptions)

Using a **machine-readable RDF ontology** instead of a text description is critical:

1. **Precision over ambiguity**: Text descriptions are inherently ambiguous. RDF provides formal, unambiguous semantics.
2. **No error multiplication**: Converting RDF → text → LLM interpretation introduces two potential error sources. Direct RDF comprehension is cleaner.
3. **Machine-readable storage**: RDF ontologies can be stored in triple stores, queried with SPARQL, and reasoned over with OWL engines.
4. **LLMs understand RDF**: GPT models have been trained on vast amounts of RDF, OWL, and Turtle syntax—they can "simulate" OWL reasoning effectively.

## Knowledge Graphs vs. Traditional Databases

**Knowledge Graphs (RDF/SPARQL)** enable:
- **Formal ontologies** that define domain semantics
- **Schema flexibility** with open-world assumptions
- **Reasoning capabilities** through RDFS/OWL inference
- **Semantic interoperability** across datasets
- **Context-aware querying** with SPARQL

This notebook showcases how ontologies transform LLMs from simple text processors into **semantic knowledge miners** that understand concepts, not just patterns.

---

Let's begin by setting up our environment and loading the necessary data.


## Step 1: Initialize OpenAI Client

First, we initialize the OpenAI client with API credentials from our environment variables. This notebook uses **GPT-5.2** for its advanced reasoning capabilities, extended context window, and deep semantic understanding.

**Why GPT-5.2?**
- **Extended context window**: Can process the entire 956K character Oliver Twist corpus in one prompt
- **Superior semantic reasoning**: Disambiguates wildlife jaguars from cars and guitars based on ontology structure
- **RDF comprehension**: Trained on extensive RDF/OWL data, enabling native understanding of ontology semantics
- **Valid Turtle generation**: Produces syntactically correct RDF that aligns with our ontology


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
# Initialize the OpenAI client 
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
 )

print("OpenAI client initialized successfully!")

# Make a call to the OpenAI client
messages = [
    {"role": "user", "content": "Tell me a coding joke"}
 ]

response = client.chat.completions.create(
    model=os.getenv("OPENAI_RESPONSES_MODEL_ID", "gpt-4o"),
    messages=messages
 )

# Print the response content
print(response.choices[0].message.content)


## Step 2: Load the Jaguar Conservation Ontology

We load the **formal ontology** that defines our domain model. This ontology contains:

- **Classes**: `Jaguar`, `Habitat`, `ConservationEffort`, `Organization`, `Threat`, etc.
- **Properties**: `hasGender`, `occursIn`, `monitoredByOrg`, `facesThreat`, etc.
- **Relationships**: How entities connect (e.g., jaguars → locations → organizations)
- **Data types**: String labels, dates, boolean values, etc.

The ontology serves as the **semantic blueprint** that guides the LLM's extraction process. It defines what concepts are relevant to our domain and how they should be structured.


In [ ]:
file_path = 'data/jaguar_ontology.ttl'
with open(file_path, 'r') as f:
    ontology = f.read()

print(f"Successfully loaded ontology from {file_path}. Content length: {len(ontology)} characters.")

## Step 3: Load the Oliver Twist Corpus

We load an **extreme challenge corpus**: Charles Dickens' complete novel "Oliver Twist" (~956,000 characters) with **jaguar conservation data scattered throughout**.

**What's in this corpus:**
- 📚 **The complete Oliver Twist novel** - authentic 19th century literature as noise
- 🐆 **Wildlife jaguar data** scattered at various points - individual jaguars (El Jefe, Macho B), locations (Arizona, Sonora), organizations (AZGFD, USFWS), threats (poaching, habitat loss)
- 🚗 **Jaguar car mentions** embedded to test disambiguation
- 🎸 **Fender Jaguar guitar references** to further test filtering

**The extreme challenge**: This isn't a curated dataset. Relevant conservation information is buried within 200,000+ tokens of Victorian fiction and deliberately placed distractors. Can GPT-5.2 find and extract ONLY the ontology-relevant wildlife data?


In [ ]:
#file_path = 'data/jaguar_corpus.txt'
file_path = 'data/oliver_twist_corpus.txt'
with open(file_path, 'r') as f:
    corpus = f.read()

print(f"Successfully loaded text from {file_path}. Content length: {len(corpus)} characters.")

## Step 4: Ontology-Aware Extraction with GPT-5.2

### The Magic: Concept Understanding Through Formal Semantics

This is where **ontology-driven extraction** happens. We provide GPT-5.2 with:
1. The **RDF ontology** (machine-readable semantic structure)
2. The **Oliver Twist corpus** (956K characters of mixed content)
3. Instructions to extract entities and relationships that **align with the ontology**

### What Makes This Remarkable?

GPT-5.2 doesn't just pattern-match "jaguar" keywords. Instead, it:
- **Maintains attention** across the entire ~250,000 token context window
- **Understands the domain** from the RDF ontology classes and properties
- **Disambiguates entities** based on semantic context (wildlife vs. cars vs. guitars)
- **Extracts ONLY relevant information** about wildlife jaguars scattered throughout the novel
- **Generates valid RDF Turtle** that conforms to our ontology structure
- **Infers relationships** between entities based on contextual clues

### Why Direct RDF Ontology (Not Text Description)?

We feed the **raw RDF ontology** directly to GPT-5.2, not a natural language description:

1. **LLMs understand RDF natively**: Trained on vast RDF/OWL corpora, GPT can parse Turtle syntax like code
2. **No lossy translation**: Converting ontology → text → LLM introduces ambiguity and errors
3. **The LLM simulates OWL reasoning**: It predicts outcomes based on training, effectively "reasoning" over the ontology
4. **Debatable "reasoning"**: Whether LLMs truly reason or pattern-match is contested, but the results speak for themselves

### Processing Time

⏱️ **Expected duration**: 1-3 minutes (with `reasoning_effort="low"` for efficiency)

### Output

The result will be **RDF Turtle code** that can be directly imported into GraphDB using "Import → Text snippet". Despite the massive, noisy corpus, the extracted data will align with your ontology—no cars, no guitars, just wildlife conservation data.

In [ ]:
#Make the prompt

prompt = f"""
Given this ontology: 
{ontology}

Extract only named entities relevant to the ontology, their relations and related information from this text. 
Think deep and analyze all information in the relevant text thoroughly. 
Try to infer relevant relationships between entities if not directly mentioned in the text.
Finally Generate RDF turtle code that align with the ontology for the found entities and relationships. 
Make sure to give all entities relevant rdfs:label.

{corpus}

"""

# Make a call to the OpenAI client
messages = [
    {"role": "user", "content": prompt}
 ]

response = client.chat.completions.create(
    model="gpt-5.2",
    messages=messages,
    max_completion_tokens=128000,
    reasoning_effort="low"
 )

# Print the response content
print(response.choices[0].message.content)

---

## Why This Requires RDF and Formal Ontologies

### The Limitations of Labeled Property Graphs (LPG)

This ontology-driven extraction approach **cannot be replicated** with LPG databases like Neo4j for fundamental architectural reasons:

### 1. **No Formal Ontologies**

**LPG databases lack formal ontology support:**
- Labels and relationship types are just **strings**, not semantically defined concepts
- No formal class hierarchies (RDFS/OWL)
- No property domain/range definitions
- No reasoning or inference capabilities

**Example:** In Neo4j, you might have:
```cypher
(:Jaguar {name: "El Jefe"})-[:OCCURS_IN]->(:Location {name: "Arizona"})
```

But the LLM has no formal structure to understand:
- What a `Jaguar` node represents (could be a car, guitar, or animal)
- What properties `Jaguar` should have
- What relationships are valid
- How concepts relate hierarchically

### 2. **No Semantic Standards**

**RDF provides W3C standards:**
- **RDFS** (Resource Description Framework Schema) - defines classes, properties, hierarchies
- **OWL** (Web Ontology Language) - enables reasoning and inference
- **SHACL** - validates data shapes
- **SKOS** - manages vocabularies and taxonomies

**LPG has:**
- Vendor-specific schemas (Neo4j, TigerGraph, etc.)
- No formal semantics
- No standardized reasoning
- No interoperability guarantees

### 3. **Knowledge Representation vs. Data Storage**

| Aspect | RDF/SPARQL (Knowledge Graphs) | LPG (Neo4j, etc.) |
|--------|-------------------------------|-------------------|
| **Schema** | Formal ontologies (RDFS/OWL) | Informal schemas |
| **Semantics** | Explicit, machine-readable | Implicit, application-level |
| **Reasoning** | Built-in (RDFS/OWL inference) | Application-specific |
| **Standards** | W3C standards (RDF, SPARQL) | Vendor-specific |
| **Interoperability** | Universal (URIs, namespaces) | Limited |
| **Open World** | Yes (absence ≠ false) | No (closed world) |
| **LLM Guidance** | Rich semantic structure | Just node/edge labels |

### 4. **Why LLMs Need Ontologies**

When extracting knowledge, the LLM requires:

✅ **With RDF Ontologies:**
- Formal class definitions (`ont:Jaguar rdfs:subClassOf ont:Animal`)
- Property domains and ranges (`ont:hasGender rdfs:domain ont:Jaguar ; rdfs:range xsd:string`)
- Relationship semantics (`ont:monitoredByOrg rdfs:range ont:Organization`)
- Hierarchical understanding (taxonomies)
- Validation rules (cardinality, data types)

❌ **With LPG:**
- Just strings: `"Jaguar"`, `"OCCURS_IN"`, `"Location"`
- No formal meaning
- No hierarchical structure
- No validation constraints
- No way to distinguish domain concepts from noise

### 5. **The "Jaguar Problem" Revisited**

**Without formal ontologies**, an LLM extracting to LPG would:
- Struggle to disambiguate jaguars (cars vs. animals)
- Have no semantic guidance on valid properties
- Create inconsistent graph structures
- Require extensive post-processing and validation

**With RDF ontologies**, the LLM:
- Understands domain semantics from formal definitions
- Generates semantically valid, structured data
- Automatically filters irrelevant entities
- Produces interoperable, standards-compliant knowledge

---

## Conclusion: Knowledge Graphs Enable Semantic AI

This notebook demonstrates how **formal ontologies** (RDFS/OWL) transform LLMs from text processors into **semantic knowledge miners**. By grounding extraction in formal semantics, we achieve:

- 🎯 **Precision** - Extract only domain-relevant entities
- 🔗 **Structure** - Generate properly connected knowledge graphs
- 🧠 **Intelligence** - Enable reasoning and inference
- 🌐 **Interoperability** - Create standards-compliant data
- 🔍 **Queryability** - Support complex SPARQL queries

**RDF + Ontologies + LLMs** = The future of intelligent knowledge extraction

LPG databases are excellent for operational graph queries, but they fundamentally lack the **semantic infrastructure** required for ontology-driven knowledge representation. For true knowledge graphs that machines can understand and reason over, RDF and formal ontologies are essential.
